# 2-Hop Retriever for Nobel Prize Research

This notebook demonstrates a 2-hop retrieval system for Nobel Prize research data:

- **First Hop**: Embedding-based search using `BAAI/bge-small-en-v1.5` to identify relevant categories (Physics, Chemistry, Medicine, Literature, Peace, Economic Sciences)
- **Second Hop**: BM25 search to retrieve specific research works within selected categories

Each chunk contains: title, authors, year, and contribution/abstract of the research work.


In [ ]:
# 1. Load and prepare Nobel Prize data from data/ directory
import re
from pathlib import Path
from typing import Dict, List

# Define Nobel Prize categories
NOBEL_CATEGORIES = [
    "Physics",
    "Chemistry",
    "Physiology or Medicine",
    "Literature",
    "Peace",
    "Economic Sciences"
]

# Map category names to file names
CATEGORY_FILE_MAP = {
    "Physics": "physics.md",
    "Chemistry": "chemistry.md",
    "Physiology or Medicine": "medicine.md",
    "Literature": "literature.md",
    "Peace": "peace.md",
    "Economic Sciences": "economics.md"
}

def parse_nobel_documents(data_dir: str = "data") -> List[Dict]:
    """Parse Nobel Prize markdown files into structured documents with category metadata."""
    documents = []
    data_path = Path(data_dir)

    if not data_path.exists():
        raise FileNotFoundError(f"Data directory '{data_dir}' not found")

    for category, filename in CATEGORY_FILE_MAP.items():
        filepath = data_path / filename
        if not filepath.exists():
            print(f"Warning: {filepath} not found, skipping {category}")
            continue

        content = filepath.read_text(encoding="utf-8")

        # Split by research work separator (---)
        sections = re.split(r'\n---\n', content)

        for section in sections:
            if not section.strip():
                continue

            # Extract title - matches "## Research: Title" or "# Title"
            title_match = re.search(r'(?:##|##)\s+Research:\s*(.+?)(?:\n|$)', section)
            if not title_match:
                title_match = re.search(r'# (.+?)(?:\n|$)', section)
            title = title_match.group(1).strip() if title_match else "Unknown"

            # Extract year - matches "**Year:** 2025"
            year_match = re.search(r'\*\*Year:\*\*\s*(\d{4})', section)
            year = int(year_match.group(1)) if year_match else None

            # Extract authors - matches "**Laureates:** Name1, Name2"
            authors_match = re.search(r'\*\*Laureates:\*\*\s*(.+?)(?:\n|$)', section)
            authors = authors_match.group(1).strip() if authors_match else "Unknown"

            # Extract contribution - matches "**Contribution:**\nText"
            contribution_match = re.search(r'\*\*Contribution:\*\*\s*\n(.*?)(?=\n\*\*|$)', section, re.DOTALL)
            contribution = contribution_match.group(1).strip() if contribution_match else ""

            # Extract significance - matches "**Significance:**\nText"
            significance_match = re.search(r'\*\*Significance:\*\*\s*\n(.*?)(?=\n\*\*|$)', section, re.DOTALL)
            significance = significance_match.group(1).strip() if significance_match else ""

            # Create full text for retrieval
            full_text = f"{title}\nYear: {year}\nAuthors: {authors}\nContribution: {contribution}\nSignificance: {significance}"

            documents.append({
                "text": full_text,
                "category": category,
                "title": title,
                "year": year,
                "authors": authors,
                "contribution": contribution,
                "significance": significance
            })

    return documents

# Load documents
documents = parse_nobel_documents("data")
print(f"Loaded {len(documents)} Nobel Prize research documents")
print(f"\nCategories available: {list(CATEGORY_FILE_MAP.keys())}")
print(f"\nSample document:")
print(f"  Category: {documents[0]['category']}")
print(f"  Title: {documents[0]['title']}")
print(f"  Year: {documents[0]['year']}")
print(f"  Authors: {documents[0]['authors'][:50]}...")


Loaded 213 Nobel Prize research documents

Categories available: ['Physics', 'Chemistry', 'Physiology or Medicine', 'Literature', 'Peace', 'Economic Sciences']

Sample document:
  Category: Physics
  Title: Nobel Prize in Physics - Research Works
  Year: None
  Authors: Unknown...


In [ ]:
# 2. Setup DSPy retrievers for 2-hop retrieval

import dspy
from dspy import Retrieve

# Configure DSPy (optional - set LM if needed)
# dspy.configure(lm=dspy.LM('openai/gpt-4o'))

# Extract texts and metadata for retrievers
texts = [doc["text"] for doc in documents]
categories = [doc["category"] for doc in documents]
metadata_list = [
    {
        "category": doc["category"],
        "title": doc["title"],
        "year": doc["year"],
        "authors": doc["authors"]
    }
    for doc in documents
]

print(f"Total documents for retrieval: {len(texts)}")
print(f"\nDocuments per category:")
from collections import Counter

cat_counts = Counter(categories)
for cat, count in cat_counts.items():
    print(f"  {cat}: {count} documents")


Total documents for retrieval: 213

Documents per category:
  Physics: 59 documents
  Chemistry: 26 documents
  Physiology or Medicine: 30 documents
  Literature: 20 documents
  Peace: 42 documents
  Economic Sciences: 36 documents


In [3]:
# 3. First Hop: BM25 Retriever for Category Selection

# Create BM25 retriever for category-level search
# We create a separate corpus for categories
category_corpus = [f"Nobel Prize in {cat}" for cat in NOBEL_CATEGORIES]

# Use dspy.Retrieve for BM25 search (default BM25)
bm25_retriever = dspy.Retrieve(k=3)

print("First Hop: BM25 Category Retriever")
print(f"Categories in corpus: {NOBEL_CATEGORIES}")
print(f"Will retrieve top {3} categories")


First Hop: BM25 Category Retriever
Categories in corpus: ['Physics', 'Chemistry', 'Physiology or Medicine', 'Literature', 'Peace', 'Economic Sciences']
Will retrieve top 3 categories


In [4]:
# 4. Second Hop: Embedding Retriever for Document Retrieval

# Setup sentence transformer for embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embedder = dspy.Embedder(model.encode)

# Create embedding retriever with all documents
# This will be used for the second hop within selected categories
doc_retriever = dspy.retrievers.Embeddings(
    embedder=embedder,
    corpus=texts,
    k=10  # Top 10 documents
)

print("Second Hop: Embedding Retriever")
print(f"Model: BAAI/bge-small-en-v1.5")
print(f"Total documents: {len(texts)}")
print(f"Will retrieve top {10} documents per query")


/data/agent_design_pattern/src/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Second Hop: Embedding Retriever
Model: BAAI/bge-small-en-v1.5
Total documents: 213
Will retrieve top 10 documents per query


In [5]:
# 5. 2-Hop Retriever Implementation (Reversed: Embedding → BM25)

class TwoHopRetriever:
    """
    A 2-hop retriever for Nobel Prize research data.

    First Hop: Uses embeddings to select relevant categories
    Second Hop: Uses BM25 to retrieve specific works from selected categories
    """

    def __init__(
        self,
        documents: List[Dict],
        k1: int = 3,  # Number of categories to select
        k2: int = 5   # Number of documents per selected category
    ):
        self.documents = documents
        self.k1 = k1
        self.k2 = k2

        # Group documents by category
        self.documents_by_category = {}
        for doc in documents:
            cat = doc["category"]
            if cat not in self.documents_by_category:
                self.documents_by_category[cat] = []
            self.documents_by_category[cat].append(doc)

        # Setup embedding model for first hop (category selection)
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer("BAAI/bge-small-en-v1.5")
        embedder = dspy.Embedder(self.model.encode)

        # Build category corpus for embedding retriever
        category_texts = [f"Nobel Prize in {cat}" for cat in NOBEL_CATEGORIES]
        self.category_retriever = dspy.retrievers.Embeddings(
            embedder=embedder,
            corpus=category_texts,
            k=k1
        )

        # Setup BM25 retriever for document-level search (second hop)
        # We store the corpus for later use
        self.all_texts = [doc["text"] for doc in documents]

    def _bm25_category_search(self, query: str, corpus: list, k: int) -> list:
        """Simple BM25-like category scoring (keyword matching)."""
        query_terms = set(query.lower().split())
        scored = []
        for passage in corpus:
            passage_lower = passage.lower()
            score = sum(1 for term in query_terms if term in passage_lower)
            scored.append((score, passage))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [p for _, p in scored[:k]]

    def retrieve(self, query: str) -> Dict:
        """
        Perform 2-hop retrieval (reversed order).

        First Hop: Embedding-based category selection
        Second Hop: BM25-based document retrieval within selected categories

        Returns:
            Dict with:
            - selected_categories: list of selected category names
            - retrieved_documents: list of retrieved documents with metadata
        """
        # First Hop: Select categories using embedding similarity
        category_texts = [f"Nobel Prize in {cat}" for cat in NOBEL_CATEGORIES]
        embedding_results = self.category_retriever(query)

        # dspy.retrievers.Embeddings returns a Prediction object with 'passages' attribute
        # Access the actual passages list from the Prediction object
        selected_categories = []
        if hasattr(embedding_results, 'passages'):
            # Prediction object - passages is a list of strings
            for cat_raw in embedding_results.passages:
                cat_name = cat_raw.replace("Nobel Prize in ", "")
                if cat_name in self.documents_by_category:
                    selected_categories.append(cat_name)
        elif isinstance(embedding_results, list):
            # Fallback: if it's a list directly
            for result in embedding_results:
                if isinstance(result, dict) and "text" in result:
                    cat_raw = result["text"]
                elif isinstance(result, str):
                    cat_raw = result
                else:
                    continue
                cat_name = cat_raw.replace("Nobel Prize in ", "")
                if cat_name in self.documents_by_category:
                    selected_categories.append(cat_name)

        # Second Hop: Retrieve documents from selected categories using BM25
        retrieved_documents = []
        for cat in selected_categories:
            if cat in self.documents_by_category:
                # Get texts and metadata for this category
                cat_docs = self.documents_by_category[cat]
                cat_texts = [doc["text"] for doc in cat_docs]

                # Use simple BM25-like scoring for document retrieval
                query_terms = set(query.lower().split())
                scored = []
                for idx, text in enumerate(cat_texts):
                    text_lower = text.lower()
                    # BM25-like score: count matching terms with position weighting
                    score = 0
                    for term in query_terms:
                        # Count occurrences of the term
                        count = text_lower.count(term)
                        # Position weighting: earlier occurrences score higher
                        first_pos = text_lower.find(term)
                        pos_weight = max(0.5, 1.0 - (first_pos / len(text_lower)))
                        score += count * pos_weight
                    scored.append((score, idx))
                scored.sort(key=lambda x: x[0], reverse=True)

                # Take top k2 documents
                for rank, (score, idx) in enumerate(scored[:self.k2]):
                    if score > 0:  # Only include documents with some relevance
                        doc = cat_docs[idx]
                        retrieved_documents.append({
                            **doc,
                            "relevance_score": score,
                            "rank": rank + 1
                        })

        return {
            "selected_categories": selected_categories,
            "retrieved_documents": retrieved_documents
        }


# Create the 2-hop retriever (reversed: embedding → BM25)
two_hop_retriever = TwoHopRetriever(
    documents=documents,
    k1=3,  # Select top 3 categories via embedding
    k2=5   # Retrieve top 5 documents per category via BM25
)

print("2-Hop Retriever initialized successfully! (Reversed: Embedding → BM25)")
print(f"Categories: {list(two_hop_retriever.documents_by_category.keys())}")
print(f"Documents per category: { {k: len(v) for k, v in two_hop_retriever.documents_by_category.items()} }")


2-Hop Retriever initialized successfully! (Reversed: Embedding → BM25)
Categories: ['Physics', 'Chemistry', 'Physiology or Medicine', 'Literature', 'Peace', 'Economic Sciences']
Documents per category: {'Physics': 59, 'Chemistry': 26, 'Physiology or Medicine': 30, 'Literature': 20, 'Peace': 42, 'Economic Sciences': 36}


In [12]:
# 6. Test the 2-Hop Retriever (Reversed: Embedding → BM25)

# Test query about Nobel Prize research
query = "Nobel Prize for peace and human rights"

print(f"Query: {query}\n")
print("=" * 80)

# Perform 2-hop retrieval
results = two_hop_retriever.retrieve(query)

print(f"\n📊 First Hop Results: Selected Categories (via Embedding)")
print("-" * 40)
for i, cat in enumerate(results["selected_categories"], 1):
    print(f"  {i}. {cat}")

print(f"\n📄 Second Hop Results: Retrieved Documents (via BM25)")
print("-" * 40)
print(f"Total documents retrieved: {len(results['retrieved_documents'])}\n")

for i, doc in enumerate(results["retrieved_documents"], 1):
    print(f"\n--- Document {i} ---")
    print(f"  Category: {doc['category']}")
    print(f"  Title: {doc['title']}")
    print(f"  Year: {doc['year']}")
    print(f"  Authors: {doc['authors']}")
    print(f"  Contribution: {doc['contribution'][:100]}...")
    print(f"  Significance: {doc['significance'][:100]}...")


Query: Nobel Prize for peace and human rights


📊 First Hop Results: Selected Categories (via Embedding)
----------------------------------------
  1. Peace
  2. Chemistry
  3. Economic Sciences

📄 Second Hop Results: Retrieved Documents (via BM25)
----------------------------------------
Total documents retrieved: 15


--- Document 1 ---
  Category: Peace
  Title: For Their Efforts to Show that Women's Rights and Their Full Participation in Peace Processes are Fundamental for Achieving Sustainable Peace
  Year: 2011
  Authors: Ellen Johnson Sirleaf, Leymah Gbowee, Tawakkol Karman
  Contribution: Sirleaf, Gbowee, and Karman fought for the safety of women and for women's rights to full participat...
  Significance: Their work has demonstrated the critical role of women in peace processes and has inspired women's r...

--- Document 2 ---
  Category: Peace
  Title: For Their Efforts to Achieve a Peaceful Solution to the Russia-Ukraine War
  Year: 2022
  Authors: Unknown
  Contribution: Bi